## query TICv8 catalog

In [ ]:
from astroquery.mast import Catalogs
import numpy as np

# Query the TIC catalog for these IDs
# We specifically want the 'GAIA' column which contains the Gaia DR2/DR3 source_id
tic_data = Catalogs.query_criteria(catalog="TIC", ID=[413738130, 116853937])
tic_data

In [ ]:
tic_data.to_pandas().columns.tolist()

In [ ]:
tic_data['GAIA'].value[1]

## query gaia DR? catalog

In [ ]:
from astroquery.gaia import Gaia

# Define your Gaia DR3 Source ID (ensure it is an integer)
gaia_id = 3448544866778118400 

# Write the ADQL query targeting your parameters
query = f"""
SELECT 
    source_id, ra, dec, parallax, pmra, pmdec, 
    phot_g_mean_mag, phot_bp_mean_mag, phot_rp_mean_mag, radial_velocity
FROM gaiadr3.gaia_source
WHERE source_id = {gaia_id}
"""

# Launch the asynchronous job
job = Gaia.launch_job_async(query)
results = job.get_results()

# Print the results table
results.pprint()

In [ ]:
df_match = results.to_pandas()
df_match.T

In [ ]:
import numpy as np

df_match['pm_diff'] =  np.sqrt((df_match['pmra'] - df_match['pmdec'])**2 + 
                               (df_match['PM RA (mas/yr)'] - df_match['PM Dec (mas/yr)'])**2)

## Perren+2023 catalog

In [ ]:
import pandas as pd

# clusters
df = pd.read_csv('../data/Perren2023/UCC_cat.csv')
df.shape

In [ ]:
df = df.rename({"name": "Name",
                #"Source": "GaiaDR3",
                "probs": "Prob",
                "RA_ICRS": "ra",
                "DE_ICRS": "dec",
                "pmRA": "pmra",
                "pmDE": "pmdec",
                "Plx": "parallax",
                "RV": "radial_velocity"
                }, axis=1)
df.columns

In [ ]:
# members
df_mem = pd.read_parquet('../data/Perren2023/UCC_members.parquet')
df_mem.shape

In [ ]:
#replace name to be compatible with other catalogs' naming formats
df_mem = df_mem.rename({"name": "Name",
                        "Source": "GaiaDR3",
                        "probs": "Prob",
                        "RA_ICRS": "ra",
                        "DE_ICRS": "dec",
                        "pmRA": "pmra",
                        "pmDE": "pmdec",
                        "Plx": "parallax",
                        "RV": "radial_velocity"
                        }, axis=1)
df_mem.columns

## crossmatch Perren catalog with target

In [ ]:
from astropy.coordinates import SkyCoord, Distance
import astropy.units as u

df_tic = tic_data.to_pandas()
coords1 = SkyCoord(ra=df_tic.ra.values*u.deg, 
                   dec=df_tic.dec.values*u.deg, 
                   distance=Distance(parallax=df_tic.plx.values*u.mas), 
                   frame='icrs')
coords2 = SkyCoord(ra=df_mem['ra'].values*u.degree, 
                   dec=df_mem['dec'].values*u.degree, 
                   distance=Distance(parallax=df_mem['parallax'].values* u.mas, 
                                     allow_negative=True),
                   frame='icrs') 

idx, d2d, d3d = coords1.match_to_catalog_sky(coords2)

In [ ]:
df_tic['sep2d_arcsec'] = d2d.to(u.arcsec).value 
df_tic['sep3d_pc'] = d3d.to(u.pc).value
df_tic.sort_values('sep2d_arcsec').head()

In [ ]:
df_mem.iloc[idx]

In [ ]:
df_mem.iloc[idx]

In [ ]:
df_match = pd.concat([df_tic.reset_index(drop=True), 
                      df_mem.iloc[idx].reset_index(drop=True)
                      ], axis=1)
df_match

## btw

In [ ]:
def get_cluster_3d_size(group):
    # 1. Create the SkyCoord object for the group
    # Note: allow_negative=True is used to handle noisy parallax data
    coords = SkyCoord(
        ra=group['ra'].values * u.degree, 
        dec=group['dec'].values * u.degree, 
        distance=Distance(parallax=group['parallax'].values * u.mas, allow_negative=True),
        frame='icrs'
    )
    
    # 2. Convert to Cartesian (x, y, z) in parsecs
    cart = coords.cartesian
    points = np.stack([cart.x.value, cart.y.value, cart.z.value], axis=1)
    
    # 3. Calculate all-to-all pairwise distances
    # For large groups, consider using a faster method like scipy.spatial.distance.pdist
    from scipy.spatial.distance import pdist
    if len(points) > 1:
        distances = pdist(points)
        return np.max(distances) #* u.pc
    return 0.0

In [ ]:
d = df_mem.groupby('Name').get_group('melotte20')
get_cluster_3d_size(d)

In [ ]:
d_cluster_size = df_mem.groupby('Name').apply(get_cluster_3d_size)
d_cluster_size

In [ ]:
d_cluster_size.median()

In [ ]:
d_cluster_size.describe()

## plot nearest clusters

In [ ]:
def plot_rdp_pmrv(
    df,
    target_gaiaid=None,
    match_id=True,
    df_target=None,
    target_label=None,
    target_color="r",
    target_ms=25,
    color="teff_val",
    marker="o",
    figsize=(10, 10),
    cmap="viridis",
):
    """
    Plot ICRS position and proper motions in 2D scatter plots,
    and parallax and radial velocity in kernel density

    Parameters
    ----------
    df : pandas.DataFrame
        contains ra, dec, parallax, pmra, pmdec, rv columns
    target_gaiaid : int
        target gaia DR2 id

    Returns
    -------
    fig : matplotlib.pyplot.Figure
    """
    assert len(df) > 0, "df is empty"
    fig, axs = pl.subplots(2, 2, figsize=figsize, constrained_layout=True)
    ax = axs.flatten()

    n = 1
    x, y = "ra", "dec"
    # _ = df.plot.scatter(x=x, y=y, c=color, marker=marker, ax=ax[n], cmap=cmap)
    errmsg = f"color={color} not in {df.columns}"
    assert color in df.columns, errmsg
    c = df[color] if color is not None else None
    cbar = ax[n].scatter(df[x], df[y], c=c, marker=marker, cmap=cmap)
    if color is not None:
        fig.colorbar(cbar, ax=ax[n], label=color)
    if target_gaiaid is not None:
        idx = df.source_id.astype(int).isin([target_gaiaid])
        if match_id:
            errmsg = f"Given cluster does not contain the target gaia id [{target_gaiaid}]"
            assert idx.any(), errmsg
            ax[n].plot(
                df.loc[idx, x],
                df.loc[idx, y],
                marker=r"$\star$",
                c=target_color,
                ms=target_ms,
                label=target_label,
            )
        else:
            assert df_target is not None, "provide df_target"
            ax[n].plot(
                df_target[x],
                df_target[y],
                marker=r"$\star$",
                c=target_color,
                ms=target_ms,
                label=target_label,
            )
    ax[n].set_xlabel("R.A. [deg]")
    ax[n].set_ylabel("Dec. [deg]")
    text = len(df[["ra", "dec"]].dropna())
    ax[n].text(0.8, 0.9, f"n={text}", fontsize=14, transform=ax[n].transAxes)
    if target_label is not None:
        ax[n].legend(loc="best")
    n = 0
    par = "parallax"
    df[par].plot.kde(ax=ax[n])
    ax[n].plot(df[par], np.full_like(df[par], -0.01), "|k", markeredgewidth=1)
    if target_gaiaid is not None:
        idx = df.source_id.astype(int).isin([target_gaiaid])
        if match_id:
            errmsg = f"Given cluster does not contain the target gaia id [{target_gaiaid}]"
            assert idx.any(), errmsg
            ax[n].axvline(
                df.loc[idx, par].values[0],
                0,
                1,
                c="k",
                ls="--",
                label=target_label,
            )
        else:
            assert df_target is not None, "provide df_target"
            ax[n].axvline(
                df_target[par], 0, 1, c="k", ls="--", label=target_label
            )

        if target_label is not None:
            ax[n].legend(loc="best")
    ax[n].set_xlabel("Parallax [mas]")
    text = len(df[par].dropna())
    ax[n].text(0.8, 0.9, f"n={text}", fontsize=14, transform=ax[n].transAxes)
    n = 3
    x, y = "pmra", "pmdec"
    # _ = df.plot.scatter(x=x, y=y, c=c, marker=marker, ax=ax[n], cmap=cmap)
    c = df[color] if color is not None else None
    cbar = ax[n].scatter(df[x], df[y], c=c, marker=marker, cmap=cmap)
    if (color is not None) & (n == 3):
        # show last colorbar only
        fig.colorbar(cbar, ax=ax[n], label=color)
    if target_gaiaid is not None:
        idx = df.source_id.astype(int).isin([target_gaiaid])
        if match_id:
            errmsg = f"Given cluster does not contain the target gaia id [{target_gaiaid}]"
            assert sum(idx) > 0, errmsg
            ax[n].plot(
                df.loc[idx, x],
                df.loc[idx, y],
                marker=r"$\star$",
                c=target_color,
                ms=target_ms,
            )
        else:
            assert df_target is not None, "provide df_target"
            ax[n].plot(
                df_target[x],
                df_target[y],
                marker=r"$\star$",
                c=target_color,
                ms="25",
            )
    ax[n].set_xlabel("PM R.A. [deg]")
    ax[n].set_ylabel("PM Dec. [deg]")
    text = len(df[["pmra", "pmdec"]].dropna())
    ax[n].text(0.8, 0.9, f"n={text}", fontsize=14, transform=ax[n].transAxes)
    n = 2
    par = "radial_velocity"
    errmsg = f"{par} is not available in {df.columns}"
    assert par in df.columns, errmsg
    try:
        df[par].plot.kde(ax=ax[n])
        ax[n].plot(
            df[par], np.full_like(df[par], -0.01), "|k", markeredgewidth=1
        )
        if target_gaiaid is not None:
            idx = df.source_id.astype(int).isin([target_gaiaid])
            if match_id:
                errmsg = f"Given cluster does not contain the target gaia id [{target_gaiaid}]"
                assert sum(idx) > 0, errmsg
                ax[n].axvline(
                    df.loc[idx, par].values[0],
                    0,
                    1,
                    c="k",
                    ls="--",
                    label=target_label,
                )
            else:
                ax[n].axvline(
                    df_target[par], 0, 1, c="k", ls="--", label=target_label
                )
        ax[n].set_xlabel("RV [km/s]")
        text = len(df[par].dropna())
        ax[n].text(
            0.8, 0.9, f"n={text}", fontsize=14, transform=ax[n].transAxes
        )
    except Exception as e:
        ax[n].clear()
        print("Error: ", e)
        npar = len(df[par].dropna())
        if npar < 10:
            errmsg = f"Cluster members have only {npar} {par} measurements."
            print("Error: ", errmsg)
            # raise ValueError(errmsg)
    return fig

In [ ]:
df_mem.iloc[idx]

In [ ]:
cname = "melotte20"
d = df_mem.query("Name==@cname")
d['source_id'] = d['GaiaDR3'].copy()
d.shape

In [ ]:
df_tic['source_id'] = df_tic['GAIA'].copy()
df_tic['parallax'] = df_tic['plx'].copy()
df_tic['pmra'] = df_tic['pmRA'].copy()
df_tic['pmdec'] = df_tic['pmDEC'].copy()

In [ ]:
import matplotlib.pyplot as pl

fig = plot_rdp_pmrv(
        d,
        match_id=False,
        target_gaiaid=int(df_tic.iloc[1].GAIA),
        df_target=df_tic.iloc[1],
        color='BP-RP'
    )